# 🔗 Módulo 3: Relaciones en SQL (PK, FK y JOINs)

**Objetivo:** Modelar el ecosistema de la Cafetería U. Sabana conectando Clientes, Productos y Ventas sin duplicar información.

### El Modelo Entidad-Relación:

1. **Tabla `clientes`:** Guarda los datos del comprador. Su Llave Primaria (PK) es `id_cliente`.

2. **Tabla `productos`:** Guarda el inventario. Su Llave Primaria (PK) es `id_producto`.

3. **Tabla `ventas`:** Es la tabla transaccional. Registra el evento de compra. Tiene su propia PK (`id_venta`), pero incluye dos Llaves Foráneas (FK): `id_cliente` e `id_producto` para saber *quién* compró *qué*.

In [1]:
# ==============================================================================
# LLAVES Y RELACIONES EN SQLITE
# ==============================================================================

import sqlite3

# 1. CONEXIÓN Y CONFIGURACIÓN
conexion = sqlite3.connect("cafeteria_relacional.db")
cursor = conexion.cursor()

# IMPORTANTE: En SQLite, debemos encender manualmente el soporte para Llaves Foráneas
cursor.execute("PRAGMA foreign_keys = ON;")

# ==============================================================================
# 2. CREACIÓN DE TABLAS INDEPENDIENTES (DIMENSIONES)
# ==============================================================================

# Tabla Clientes (PK: id_cliente)
cursor.execute('''
    CREATE TABLE IF NOT EXISTS clientes (
        id_cliente INTEGER PRIMARY KEY,
        nombre TEXT NOT NULL,
        tipo TEXT NOT NULL
    )
''')

# Tabla Productos (PK: id_producto)
cursor.execute('''
    CREATE TABLE IF NOT EXISTS productos (
        id_producto INTEGER PRIMARY KEY AUTOINCREMENT,
        nombre TEXT NOT NULL,
        precio REAL NOT NULL
    )
''')

# ==============================================================================
# 3. CREACIÓN DE TABLA TRANSACCIONAL (HECHOS) CON LLAVES FORÁNEAS
# ==============================================================================

# Tabla Ventas (PK: id_venta | FK: id_cliente, id_producto)
cursor.execute('''
    CREATE TABLE IF NOT EXISTS ventas (
        id_venta INTEGER PRIMARY KEY AUTOINCREMENT,
        fecha TEXT DEFAULT CURRENT_TIMESTAMP,
        cantidad INTEGER NOT NULL,
        id_cliente INTEGER,
        id_producto INTEGER,
        
        FOREIGN KEY (id_cliente) REFERENCES clientes (id_cliente),
        FOREIGN KEY (id_producto) REFERENCES productos (id_producto)
    )
''')

conexion.commit()

print("✅ Tablas creadas y relacionadas exitosamente.")

# ==============================================================================
# 4. INSERTAR DATOS (POBLAR LA BASE DE DATOS)
# ==============================================================================
# Limpiamos tablas para el ejemplo
cursor.execute('DELETE FROM ventas')
cursor.execute('DELETE FROM clientes')
cursor.execute('DELETE FROM productos')

# Insertamos Clientes
cursor.execute("INSERT INTO clientes (id_cliente, nombre, tipo) VALUES (1001, 'Diego Zuluaga', 'Profesor')")
cursor.execute("INSERT INTO clientes (id_cliente, nombre, tipo) VALUES (1002, 'Ana Gómez', 'Estudiante')")

# Insertamos Productos
cursor.execute("INSERT INTO productos (id_producto, nombre, precio) VALUES (1, 'Café Tostao', 5000)")
cursor.execute("INSERT INTO productos (id_producto, nombre, precio) VALUES (2, 'Chocolatina Jet', 1200)")

# Insertamos Ventas (Usando solo los IDs, sin repetir nombres)
# Diego (1001) compra 2 Cafés (1)
cursor.execute("INSERT INTO ventas (id_cliente, id_producto, cantidad) VALUES (1001, 1, 2)")
# Ana (1002) compra 5 Chocolatinas (2)
cursor.execute("INSERT INTO ventas (id_cliente, id_producto, cantidad) VALUES (1002, 2, 5)")

conexion.commit()
print("➕ Datos insertados correctamente.")

# ==============================================================================
# 5. CONSULTAS AVANZADAS: LA CLÁUSULA JOIN
# ==============================================================================
# Si hacemos un SELECT a ventas, solo veremos números (IDs). 
# Usamos JOIN para cruzar las tablas y traer los nombres reales.

consulta_sql = '''
    SELECT 
        ventas.id_venta,
        ventas.fecha,
        clientes.nombre AS nombre_cliente,
        clientes.tipo AS tipo_cliente,
        productos.nombre AS nombre_producto,
        ventas.cantidad,
        (productos.precio * ventas.cantidad) AS total_pagado
    FROM ventas
    JOIN clientes ON ventas.id_cliente = clientes.id_cliente
    JOIN productos ON ventas.id_producto = productos.id_producto
'''

cursor.execute(consulta_sql)

reporte_ventas = cursor.fetchall()

print("\n📊 REPORTE DE VENTAS CONSOLIDADO (JOIN):")
print("-" * 80)
for fila in reporte_ventas:
    print(f"Venta #{fila[0]} | Fecha: {fila[1][:10]}")
    print(f"👤 Cliente: {fila[2]} ({fila[3]})")
    print(f"🛒 Producto: {fila[5]}x {fila[4]} | Total: ${fila[6]:,.2f}")
    print("-" * 80)

conexion.close()

✅ Tablas creadas y relacionadas exitosamente.
➕ Datos insertados correctamente.

📊 REPORTE DE VENTAS CONSOLIDADO (JOIN):
--------------------------------------------------------------------------------
Venta #3 | Fecha: 2026-03-25
👤 Cliente: Diego Zuluaga (Profesor)
🛒 Producto: 2x Café Tostao | Total: $10,000.00
--------------------------------------------------------------------------------
Venta #4 | Fecha: 2026-03-25
👤 Cliente: Ana Gómez (Estudiante)
🛒 Producto: 5x Chocolatina Jet | Total: $6,000.00
--------------------------------------------------------------------------------


# 🔗 Tarea Clase 7: Arquitectura Relacional - Módulo de Proveedores

Descripción de la Actividad:

Para garantizar la integridad de los datos en nuestro sistema "Cafetería U. Sabana", debemos aplicar las reglas de Normalización y el uso de Llaves (PK y FK). En esta tarea, integrarán a los proveedores en el modelo relacional.

Instrucciones de Autogestión:

    Diseño Lógico (Markdown): Expliquen cómo se relaciona un Proveedor con un Producto. (Ej: Un proveedor suministra muchos productos. Por lo tanto, la tabla productos debe tener una Llave Foránea que apunte al proveedor).

    Creación de Tabla proveedores: Creen la tabla en SQLite con su Llave Primaria (nit), nombre_empresa y ciudad.

    Modificación de Tabla productos: Alteren o recreen la tabla productos para que incluya la columna nit_proveedor como Llave Foránea (FK).

    Inserción de Datos: Inserten 2 proveedores y 3 productos, asignando a cada producto el NIT de su respectivo proveedor.

    Consulta JOIN: Escriban una consulta SQL que cruce ambas tablas e imprima un reporte que muestre: Nombre del Producto | Precio | Nombre del Proveedor que lo suministra.

Requisitos de Entrega:

    Archivo .ipynb con el código funcional y documentado.

    Asegúrense de incluir el comando PRAGMA foreign_keys = ON; al inicio de su conexión.

“Las bases de datos relacionales son el cimiento sobre el cual se construyen las decisiones empresariales confiables.”